In [ ]:
from solver.energy import TemperatureModule
from solver.simple import SIMPLEcfd
from core.mesh import Mesh
from core.settings import SolveSettings, PlotSettings
from core.fields import FlowField, FaceField
import pprint

import numpy as np

In [ ]:
x = 0.9*1e-3# in [m]
y = 0.5*1e-3
z = 0.5*1e-3
nx = 50
nz = 50
dt = 1e-6

mesh: Mesh = Mesh(x, y, z, nx, 1, nz, dt)
solverSettings = SolveSettings(0.25, 50)
plotSettings = PlotSettings()


def main(solveSettings: SolveSettings, plotSettings: PlotSettings, mesh: Mesh):
    Nq = mesh.Nx
    Nz = mesh.Nz
    fields = FlowField(mesh, Nq, Nz)
    faces = FaceField(mesh)
    energy = TemperatureModule(solveSettings, fields, faces, mesh, mesh.dt)
    simpleSolve =  SIMPLEcfd(solveSettings, fields, faces, mesh, mesh.dt)    

    t = 0.0
    steady_state = False
    depth = -1
    x=0
    n=100
    while x < n:
        print(f"Entered main loop, t = {t}")
        fields.update_material_properties()
        energy.laser_update(current_t = t)
        fields.update_material_properties()
        simpleSolve.run(50)
        energy.solve_energy()
        depthNew = energy.compute_depth()
        t += mesh.dt

        if depthNew == depth:
            print("reached S.S.")
            steady_state = True
        else:
            depth = depthNew
        x += 1
    pprint.pprint(fields.T)
    return fields

In [ ]:
F = main(
    solveSettings= solverSettings,
    plotSettings= plotSettings,
    mesh = mesh,
)


In [ ]:
import matplotlib.pyplot as plt
from physics.heat_source import LaserSource

F.T = np.ones_like(F.T) * 303
laser = LaserSource(
                P = 500,    #[W]
                A = 0.8,    #[1]
                Cp = F.cp,  #[J/kg-K]
                a = 100*1e-6,    #[m] -> micrometer * 1e-6
                b = 100*1e-6,    #[m] ""
                c = 30 * 1e-6,   #[m] ""
                rho=F.rho,  #[kg/m3]
                k=F.k,      #[W/m3-K]
                V = 0.5,      #[m/s]
                t_prime = 0,
        )

dT = laser.temperature_update(
    fields          = F, 
    mesh            = mesh, 
    current_time    = 0.001, 
    delta_time      = 1e-4
)


plt.figure()
plt.pcolormesh(mesh.x, mesh.z, dT, shading="auto")
plt.colorbar(label="Temperature increment [K]")
plt.xlabel("x [m]")
plt.ylabel("z [m]")
plt.gca().invert_yaxis()
plt.title("Laser temperature increment")
# plt.show()

In [ ]:
plt.matshow(F.T)